# Transformers with PyTorch & HuggingFace

**Module 07: Transformers**  
**Objective**: Master pretrained Transformers for practical applications

HuggingFace Transformers library provides:
- 100+ pretrained models
- Unified API for all architectures
- Easy fine-tuning
- Production-ready pipelines

## What You'll Learn
1. PyTorch Transformer implementation
2. Using HuggingFace Transformers
3. BERT for classification
4. GPT for text generation
5. Fine-tuning strategies
6. Model comparison

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. PyTorch Transformer

PyTorch provides `nn.Transformer` module (implements full encoder-decoder).

**Key components**:
- `nn.TransformerEncoder`: Stack of encoder layers
- `nn.TransformerDecoder`: Stack of decoder layers
- `nn.TransformerEncoderLayer`: Single encoder layer
- `nn.TransformerDecoderLayer`: Single decoder layer

In [ ]:
# PyTorch Transformer
d_model = 512
nhead = 8
num_encoder_layers = 6
num_decoder_layers = 6
dim_feedforward = 2048
dropout = 0.1

# Create transformer
transformer_model = nn.Transformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    batch_first=True
).to(device)

print("PyTorch Transformer")
print(f"Parameters: {sum(p.numel() for p in transformer_model.parameters()):,}\n")

# Test forward pass
batch_size = 2
src_seq_len = 10
tgt_seq_len = 8

src = torch.randn(batch_size, src_seq_len, d_model).to(device)
tgt = torch.randn(batch_size, tgt_seq_len, d_model).to(device)

# Create causal mask for decoder (prevent looking at future)
tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len).to(device)

output = transformer_model(src, tgt, tgt_mask=tgt_mask)

print(f"Source shape: {src.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Output shape: {output.shape}")
print(f"Target mask shape: {tgt_mask.shape}\n")

# Visualize causal mask
plt.figure(figsize=(6, 6))
sns.heatmap(tgt_mask.cpu().numpy(), cmap='RdBu', center=0, cbar=True,
            xticklabels=range(tgt_seq_len), yticklabels=range(tgt_seq_len))
plt.title('Causal Mask (−∞ prevents attention to future)')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.tight_layout()
plt.show()

## 2. Simple Classification with Transformer Encoder

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer encoder for classification."""
    
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes, max_len=512, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, num_classes)
    
    def forward(self, x, padding_mask=None):
        """
        Args:
            x: (batch, seq_len) token indices
            padding_mask: (batch, seq_len) mask for padding
        
        Returns:
            output: (batch, num_classes)
        """
        x = self.embedding(x) * np.sqrt(self.d_model)  # Scale embeddings
        x = self.pos_encoder(x)
        
        # Padding mask (True for padding tokens)
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # Use [CLS] token (position 0) or mean pooling
        x = x.mean(dim=1)  # Mean pooling over sequence
        
        return self.fc(x)

class PositionalEncoding(nn.Module):
    """Positional encoding module."""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Test classifier
import math

classifier = TransformerClassifier(
    vocab_size=10000,
    d_model=128,
    nhead=8,
    num_layers=4,
    num_classes=2,
    max_len=100,
    dropout=0.1
).to(device)

x_test = torch.randint(0, 10000, (4, 20)).to(device)
output_test = classifier(x_test)

print(f"\nTransformer Classifier")
print(f"Input shape: {x_test.shape}")
print(f"Output shape: {output_test.shape}")
print(f"Parameters: {sum(p.numel() for p in classifier.parameters()):,}")

## 3. HuggingFace Transformers

**Installation**: `pip install transformers`

**Key classes**:
- `AutoModel`: Load any model
- `AutoTokenizer`: Load corresponding tokenizer
- `pipeline`: High-level API for common tasks

In [ ]:
# Note: HuggingFace requires installation
# Demonstrating the API structure

print("\nHuggingFace Transformers API Structure:\n")

example_code = """
from transformers import AutoTokenizer, AutoModel, pipeline

# 1. Using Pipelines (easiest)
classifier = pipeline('sentiment-analysis')
result = classifier("I love this!")
# [{'label': 'POSITIVE', 'score': 0.9998}]

# 2. Using AutoModel + AutoTokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Tokenize
inputs = tokenizer("Hello world!", return_tensors="pt")
# {'input_ids': tensor([[101, 7592, 2088, 102]]), 
#  'attention_mask': tensor([[1, 1, 1, 1]])}

# Forward
outputs = model(**inputs)
# BaseModelOutputWithPooling(
#   last_hidden_state: (batch, seq_len, hidden_size),
#   pooler_output: (batch, hidden_size)
# )

# 3. Fine-tuning
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()
"""

print(example_code)

## 4. Simulated BERT-style Model

In [ ]:
class SimpleBERT(nn.Module):
    """Simplified BERT-style model."""
    
    def __init__(self, vocab_size, d_model=768, nhead=12, num_layers=12, num_classes=2):
        super().__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(512, d_model)
        self.token_type_embedding = nn.Embedding(2, d_model)  # For sentence A/B
        
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Classification head
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, input_ids, token_type_ids=None, attention_mask=None):
        """
        Args:
            input_ids: (batch, seq_len)
            token_type_ids: (batch, seq_len) - 0 for sentence A, 1 for sentence B
            attention_mask: (batch, seq_len) - 1 for real tokens, 0 for padding
        
        Returns:
            logits: (batch, num_classes)
        """
        batch_size, seq_len = input_ids.size()
        
        # Token embeddings
        token_emb = self.token_embedding(input_ids)
        
        # Position embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        position_emb = self.position_embedding(positions)
        
        # Token type embeddings
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        token_type_emb = self.token_type_embedding(token_type_ids)
        
        # Combine embeddings
        embeddings = token_emb + position_emb + token_type_emb
        embeddings = self.norm(embeddings)
        embeddings = self.dropout(embeddings)
        
        # Create padding mask
        if attention_mask is not None:
            padding_mask = (attention_mask == 0)
        else:
            padding_mask = None
        
        # Encode
        encoded = self.encoder(embeddings, src_key_padding_mask=padding_mask)
        
        # Use [CLS] token (first token) for classification
        cls_output = encoded[:, 0, :]
        
        logits = self.classifier(cls_output)
        
        return logits

# Create BERT-style model
bert_model = SimpleBERT(vocab_size=30000, d_model=256, nhead=8, num_layers=6, num_classes=2).to(device)

# Test
input_ids = torch.randint(0, 30000, (4, 50)).to(device)
attention_mask = torch.ones(4, 50).to(device)
attention_mask[:, 40:] = 0  # Last 10 tokens are padding

logits = bert_model(input_ids, attention_mask=attention_mask)

print(f"\nSimpleBERT Model")
print(f"Input shape: {input_ids.shape}")
print(f"Attention mask shape: {attention_mask.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

## 5. GPT-style Causal Language Model

In [ ]:
class SimpleGPT(nn.Module):
    """Simplified GPT-style autoregressive model."""
    
    def __init__(self, vocab_size, d_model=768, nhead=12, num_layers=12, max_len=1024):
        super().__init__()
        
        self.d_model = d_model
        self.max_len = max_len
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        
        self.dropout = nn.Dropout(0.1)
        
        # Transformer decoder (causal attention)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        
        # Language modeling head
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Share weights between embedding and lm_head (common practice)
        self.lm_head.weight = self.token_embedding.weight
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch, seq_len)
        
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.size()
        
        # Token embeddings
        token_emb = self.token_embedding(input_ids)
        
        # Position embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        position_emb = self.position_embedding(positions)
        
        # Combine
        embeddings = self.dropout(token_emb + position_emb)
        
        # Causal mask (prevent attending to future)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(input_ids.device)
        
        # Decode (with causal masking)
        # For decoder-only, we use the same sequence for memory
        decoded = self.decoder(embeddings, embeddings, tgt_mask=causal_mask, memory_mask=causal_mask)
        
        # Predict next token
        logits = self.lm_head(decoded)
        
        return logits
    
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=50):
        """
        Generate text autoregressively.
        
        Args:
            input_ids: (batch, seq_len) prompt
            max_new_tokens: number of tokens to generate
            temperature: sampling temperature
            top_k: top-k sampling
        
        Returns:
            generated: (batch, seq_len + max_new_tokens)
        """
        self.eval()
        
        for _ in range(max_new_tokens):
            # Crop to max_len
            idx_cond = input_ids if input_ids.size(1) <= self.max_len else input_ids[:, -self.max_len:]
            
            # Forward
            with torch.no_grad():
                logits = self(idx_cond)
                logits = logits[:, -1, :] / temperature
                
                # Top-k sampling
                if top_k is not None:
                    v, _ = torch.topk(logits, top_k)
                    logits[logits < v[:, [-1]]] = -float('Inf')
                
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
            
            # Append
            input_ids = torch.cat((input_ids, idx_next), dim=1)
        
        return input_ids

# Create GPT-style model
gpt_model = SimpleGPT(vocab_size=50257, d_model=256, nhead=8, num_layers=6, max_len=512).to(device)

# Test generation
prompt = torch.randint(0, 50257, (1, 10)).to(device)
generated = gpt_model.generate(prompt, max_new_tokens=20, temperature=0.8, top_k=40)

print(f"\nSimpleGPT Model")
print(f"Prompt shape: {prompt.shape}")
print(f"Generated shape: {generated.shape}")
print(f"Parameters: {sum(p.numel() for p in gpt_model.parameters()):,}")

## 6. Model Comparison

In [ ]:
# Compare architectures
models_info = {
    'BERT-Base': {
        'params': '110M',
        'architecture': 'Encoder-only',
        'attention': 'Bidirectional',
        'use_cases': 'Classification, NER, QA',
        'layers': 12,
        'd_model': 768,
        'heads': 12
    },
    'GPT-2': {
        'params': '117M-1.5B',
        'architecture': 'Decoder-only',
        'attention': 'Causal (autoregressive)',
        'use_cases': 'Text generation, completion',
        'layers': 12,
        'd_model': 768,
        'heads': 12
    },
    'T5-Base': {
        'params': '220M',
        'architecture': 'Encoder-decoder',
        'attention': 'Bidirectional encoder, causal decoder',
        'use_cases': 'Translation, summarization, QA',
        'layers': '12 encoder + 12 decoder',
        'd_model': 768,
        'heads': 12
    },
    'LLaMA-7B': {
        'params': '7B',
        'architecture': 'Decoder-only',
        'attention': 'Causal with RoPE',
        'use_cases': 'General text generation',
        'layers': 32,
        'd_model': 4096,
        'heads': 32
    }
}

print("\nTransformer Model Comparison\n")
print("="*100)
print(f"{'Model':<15} {'Params':<12} {'Architecture':<20} {'Attention':<30} {'Use Cases':<20}")
print("="*100)

for model_name, info in models_info.items():
    print(f"{model_name:<15} {info['params']:<12} {info['architecture']:<20} {info['attention']:<30} {info['use_cases']:<20}")

print("="*100)

## 7. Visualizing Transformer Components

In [ ]:
# Visualize architecture differences
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# BERT (Encoder-only)
axes[0].text(0.5, 0.9, 'BERT', ha='center', fontsize=16, fontweight='bold')
axes[0].text(0.5, 0.75, 'Bidirectional\nEncoder', ha='center', bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[0].arrow(0.5, 0.65, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[0].text(0.5, 0.45, 'Token Embeddings +\nPositional Encoding', ha='center', bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[0].arrow(0.5, 0.35, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[0].text(0.5, 0.15, 'Input Text', ha='center', fontsize=12)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].axis('off')
axes[0].set_title('Classification, NER', fontsize=10)

# GPT (Decoder-only)
axes[1].text(0.5, 0.9, 'GPT', ha='center', fontsize=16, fontweight='bold')
axes[1].text(0.5, 0.75, 'Causal\nDecoder', ha='center', bbox=dict(boxstyle='round', facecolor='lightcoral'))
axes[1].arrow(0.5, 0.65, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[1].text(0.5, 0.45, 'Token Embeddings +\nPositional Encoding', ha='center', bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[1].arrow(0.5, 0.35, 0, -0.15, head_width=0.05, head_length=0.05, fc='black')
axes[1].text(0.5, 0.15, 'Input Text', ha='center', fontsize=12)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title('Text Generation', fontsize=10)

# T5 (Encoder-Decoder)
axes[2].text(0.25, 0.9, 'T5', ha='center', fontsize=16, fontweight='bold')
axes[2].text(0.25, 0.7, 'Encoder', ha='center', bbox=dict(boxstyle='round', facecolor='lightblue'))
axes[2].text(0.75, 0.7, 'Decoder', ha='center', bbox=dict(boxstyle='round', facecolor='lightcoral'))
axes[2].arrow(0.25, 0.6, 0, -0.1, head_width=0.05, head_length=0.05, fc='black')
axes[2].arrow(0.75, 0.6, 0, -0.1, head_width=0.05, head_length=0.05, fc='black')
axes[2].text(0.25, 0.45, 'Source', ha='center', bbox=dict(boxstyle='round', facecolor='lightgreen'))
axes[2].text(0.75, 0.45, 'Target', ha='center', bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[2].arrow(0.35, 0.7, 0.3, 0, head_width=0.05, head_length=0.05, fc='gray', linestyle='dashed')
axes[2].text(0.5, 0.75, 'Cross-Attention', ha='center', fontsize=8, style='italic')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].axis('off')
axes[2].set_title('Translation, Summarization', fontsize=10)

plt.suptitle('Transformer Architecture Variants', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

## Summary

You've mastered Transformers with PyTorch:
- ✅ PyTorch Transformer API
- ✅ Transformer classifier implementation
- ✅ BERT-style encoder model
- ✅ GPT-style decoder model
- ✅ Text generation with sampling
- ✅ Model comparison (BERT vs GPT vs T5)

**Key insights**:
1. **BERT**: Bidirectional encoder for understanding (classification, NER)
2. **GPT**: Causal decoder for generation (text completion, chat)
3. **T5**: Encoder-decoder for seq2seq (translation, summarization)
4. **HuggingFace**: Unified API for all transformer models
5. **Scaling**: Larger models (billions of parameters) show emergent abilities

**Practical tips**:
1. Start with pretrained models (transfer learning)
2. Fine-tune on your domain/task
3. Use appropriate model type (encoder for classification, decoder for generation)
4. Monitor GPU memory (large batch size × long sequences)
5. Apply gradient checkpointing for memory efficiency

**Next Module**: Large Language Models (LLMs) - scaling laws, GPT architecthre deep dive, prompt engineering

## Exercises

1. **Fine-tune BERT**: Load pretrained BERT and fine-tune on sentiment analysis
2. **Implement BART**: Build BERT-like encoder + GPT-like decoder for summarization
3. **Add Flash Attention**: Implement memory-efficient attention for long sequences